# Phase 1 — LLM Basics & Local Dev Exploration

This notebook walks through the key Phase 1 patterns: calling Ollama locally via the OpenAI SDK,
building typed agent configs with Pydantic v2, running concurrent async requests, and comparing
three prompt engineering iterations on the same task.

## 1. Calling Ollama Locally

Ollama exposes an OpenAI-compatible REST API at `http://localhost:11434/v1`. We can use the
standard `openai` SDK — both sync and async clients — without changing any application code.

In [ ]:
from __future__ import annotations

import asyncio

import openai

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"  # pull with: ollama pull llama3.2

# ── Sync client ──────────────────────────────────────────────────────────────
sync_client = openai.OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

response = sync_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: hello from ollama"}],
    max_tokens=20,
    temperature=0.0,
)
print("Sync:", response.choices[0].message.content)
# Expected: hello from ollama

# ── Async client ─────────────────────────────────────────────────────────────
async_client = openai.AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


async def async_ping() -> str:
    resp = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: async ok"}],
        max_tokens=10,
        temperature=0.0,
    )
    return resp.choices[0].message.content or ""


# In a Jupyter kernel the event loop is already running; use await directly.
result = await async_ping()
print("Async:", result)
# Expected: async ok

## 2. Pydantic v2 Agent Config

`projects/phase1-foundations/config.py` defines `AgentConfig` — a Pydantic v2 model with
discriminated union provider selection, field validators, and JSON schema export. Here we
import it directly and exercise the main patterns.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Add the phase1 directory to sys.path so we can import config directly.
phase1_dir = str(Path.cwd().parent / "projects" / "phase1-foundations")
if phase1_dir not in sys.path:
    sys.path.insert(0, phase1_dir)

from config import AgentConfig, OllamaProviderConfig, OpenAIProviderConfig, RetryConfig

# Build a local Ollama agent
local_agent = AgentConfig(
    name="local-research-agent",
    provider=OllamaProviderConfig(model="llama3.2"),
    temperature=0.3,
    system_prompt="You are a precise research assistant. Always cite sources.",
)
print("Provider:", local_agent.provider_name, " Model:", local_agent.model_name)
# Provider: ollama  Model: llama3.2

# Build a cloud agent with custom retry policy
cloud_agent = AgentConfig(
    name="cloud-summarizer",
    provider=OpenAIProviderConfig(model="gpt-4o-mini", api_key="sk-placeholder"),
    temperature=0.5,
    max_tokens=512,
    retry=RetryConfig(max_attempts=5, backoff_seconds=2.0),
)
print(local_agent.model_dump_json(indent=2))

# Inspect the generated JSON schema — useful for tool/API documentation
schema = AgentConfig.model_json_schema()
print("\nTop-level schema keys:", list(schema.keys()))
# ['title', 'description', 'type', 'properties', 'required', '$defs']

## 3. Async Concurrent Calls

`asyncio.gather` fires multiple coroutines simultaneously. Against a local Ollama instance
this can halve wall-clock time when the model supports concurrent requests. The demo below
uses `unittest.mock` so it runs offline — swap in real `async_client.chat.completions.create`
calls when Ollama is running.

In [ ]:
from __future__ import annotations

import asyncio
import time
from unittest.mock import AsyncMock, MagicMock

# Build a mock async client that simulates a 0.1 s round-trip.
def _make_mock_client(reply: str) -> AsyncMock:
    choice = MagicMock()
    choice.message.content = reply
    completion = MagicMock()
    completion.choices = [choice]
    mock_create = AsyncMock(return_value=completion)
    client = MagicMock()
    client.chat.completions.create = mock_create
    return client


async def fetch(client: object, prompt: str) -> str:
    await asyncio.sleep(0.1)  # simulated latency
    resp = await client.chat.completions.create(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.0,
    )
    return resp.choices[0].message.content


client_a = _make_mock_client("Summary: RAG reduces hallucination.")
client_b = _make_mock_client("Keywords: retrieval, embeddings, reranking.")

start = time.perf_counter()
summary, keywords = await asyncio.gather(
    fetch(client_a, "Summarise RAG in one sentence."),
    fetch(client_b, "List three RAG keywords."),
)
elapsed = time.perf_counter() - start

print(f"Both finished in {elapsed:.2f}s (concurrent, not {2 * 0.1:.2f}s sequential)")
print("Summary: ", summary)
print("Keywords:", keywords)
# Both finished in ~0.10s (concurrent, not 0.20s sequential)

## 4. Prompt Engineering Comparison

`projects/phase1-foundations/prompt_engineering.py` defines three prompt variants for the same
task — extracting structured action items from a meeting transcript.

| Variant | Strategy | Expected quality |
|---------|----------|------------------|
| V1 | Bare user prompt | Inconsistent, verbose |
| V2 | System prompt with role + format instruction | Structured, consistent |
| V3 | System prompt + two few-shot examples | Tight format, best on small models |

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

phase1_dir = str(Path.cwd().parent / "projects" / "phase1-foundations")
if phase1_dir not in sys.path:
    sys.path.insert(0, phase1_dir)

from prompt_engineering import TRANSCRIPT, V1_USER, V2_SYSTEM, V2_USER, V3_USER

SEP = "-" * 60

print("TRANSCRIPT (shared across all variants):")
print(TRANSCRIPT)

print(SEP)
print("V1 — bare user prompt (no system, no examples):")
print(V1_USER[:200], "...")

print(SEP)
print("V2 — system prompt role + format instruction:")
print("SYSTEM:", V2_SYSTEM[:120], "...")
print("USER:  ", V2_USER[:80], "...")

print(SEP)
print("V3 — system prompt + two few-shot examples (first 300 chars of user msg):")
print(V3_USER[:300], "...")

# To actually call Ollama, uncomment:
# from prompt_engineering import main
# await main()